# Load XUI front-end results into the central audit tables

Reads `xui_link_results.json` produced locally by `check_case_links.py`, then:

1. Writes a run row to **test_automation_runs2** and one result row per case to
   **test_automation_results2** (same schema as the other case-linking notebooks,
   so it shows up in Cross Coverage / Trends).
2. Writes a detailed **case_link_xui_evidence** table (ref, kind, direction,
   expected/shown, missing/extra, reason issues, message) for evidence.

### Getting the file in
Run `check_case_links.py` locally, then upload `XUI_TESTS/results/xui_link_results.json`
to your Workspace Results folder (default path below) — drag-drop in the Workspace UI,
or copy into `/Workspace/Users/<you>/Results/Case_Linking_Tests/`.

In [0]:
import os, json, uuid
from datetime import datetime
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, TimestampType)

run_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
test_results_path = "/Workspace/Users/" + run_user + "/Results/Case_Linking_Tests"

dbutils.widgets.text("results_json_path", test_results_path + "/xui_link_results.json")
dbutils.widgets.text("run_tag", "front_end")
results_json_path = dbutils.widgets.get("results_json_path")
run_tag = dbutils.widgets.get("run_tag") or "front_end"

STATE_UNDER_TEST = "caseLinking_xui"
AUTOMATION_NAME  = "Case_Linking_XUI_Tests"
RUNS_TABLE       = "test_automation_runs2"
RESULTS_TABLE    = "test_automation_results2"
EVIDENCE_TABLE   = "case_link_xui_evidence"

print(f"User: {run_user}")
print(f"Reading: {results_json_path}")

In [0]:
if not os.path.exists(results_json_path):
    raise FileNotFoundError(
        f"{results_json_path} not found. Run check_case_links.py locally and upload "
        f"results/xui_link_results.json to your Results folder (or set the results_json_path widget).")

with open(results_json_path, encoding="utf-8") as f:
    payload = json.load(f)

cases = payload.get("results", [])
print(f"env={payload.get('env')}  run_ts={payload.get('run_ts')}  run_status={payload.get('run_status')}")
print(f"total={payload.get('total')}  pass={payload.get('passed')}  fail={payload.get('failed')}  error={payload.get('errored')}")
print(f"cases in file: {len(cases)}")

In [0]:
def _join(v):
    return "|".join(v) if isinstance(v, list) else (v or "")

evidence_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("run_ts", StringType(), True),
    StructField("env", StringType(), True),
    StructField("ccd_ref", StringType(), True),
    StructField("aria_case_no", StringType(), True),
    StructField("kind", StringType(), True),
    StructField("direction", StringType(), True),
    StructField("status", StringType(), True),
    StructField("expected", IntegerType(), True),
    StructField("shown", IntegerType(), True),
    StructField("missing", StringType(), True),
    StructField("extra", StringType(), True),
    StructField("reason_issues", StringType(), True),
    StructField("message", StringType(), True),
    StructField("evidence_file", StringType(), True),
])

run_id = str(uuid.uuid4())
run_ts = payload.get("run_ts")
env    = payload.get("env")

ev_rows = []
for c in cases:
    ev_rows.append((
        run_id, run_ts, env,
        str(c.get("ccd_ref", "")), str(c.get("aria_case_no") or ""),
        c.get("kind", "positive"), c.get("direction", "to"), c.get("status", ""),
        int(c.get("expected", 0)), int(c.get("shown", 0)),
        _join(c.get("missing")), _join(c.get("extra")), _join(c.get("reason_issues")),
        c.get("message", ""),
        c.get("evidence", ""),
    ))

evidence_df = spark.createDataFrame(ev_rows, schema=evidence_schema)
display(evidence_df)

In [0]:
evidence_df.write.mode("append").option("mergeSchema", "true").saveAsTable(EVIDENCE_TABLE)
print(f"Wrote {evidence_df.count()} rows to {EVIDENCE_TABLE} (run_id={run_id})")

In [0]:
runs_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("run_user", StringType(), True),
    StructField("run_start_datetime", TimestampType(), True),
    StructField("run_end_datetime", TimestampType(), True),
    StructField("run_by_automation_name", StringType(), True),
    StructField("run_tag", StringType(), True),
    StructField("run_status", StringType(), True),
    StructField("state_under_test", StringType(), True),
])
results_schema = StructType([
    StructField("result_id", StringType(), False),
    StructField("run_id", StringType(), False),
    StructField("test_name", StringType(), True),
    StructField("test_field", StringType(), True),
    StructField("test_from_state", StringType(), True),
    StructField("status", StringType(), True),
    StructField("message", StringType(), True),
])

run_status = payload.get("run_status") or (
    "FAILED" if any(str(c.get("status", "")).upper() in ("FAIL", "ERROR") for c in cases) else "PASSED")
now = datetime.utcnow()

spark.createDataFrame(
    [(run_id, run_user, now, now, AUTOMATION_NAME, run_tag, run_status, STATE_UNDER_TEST)],
    schema=runs_schema,
).write.mode("append").option("mergeSchema", "true").saveAsTable(RUNS_TABLE)

result_rows = []
for c in cases:
    detail = (f"[{c.get('kind','positive')}/{c.get('direction','to')}] "
              f"expected={c.get('expected',0)} shown={c.get('shown',0)}"
              + (f" | {c.get('message','')}" if c.get('message') else ""))
    result_rows.append((
        str(uuid.uuid4()), run_id,
        f"xui_link_check.{c.get('kind','positive')}",
        str(c.get("ccd_ref", "")),
        STATE_UNDER_TEST,
        str(c.get("status", "")),
        detail[:8000],
    ))

result_rows.append((
    str(uuid.uuid4()), run_id, "xui_link_check.summary", "ALL", STATE_UNDER_TEST,
    run_status, (f"total={payload.get('total')} pass={payload.get('passed')} "
                 f"fail={payload.get('failed')} error={payload.get('errored')}")[:8000],
))

spark.createDataFrame(result_rows, schema=results_schema)      .write.mode("append").option("mergeSchema", "true").saveAsTable(RESULTS_TABLE)

print(f"run_id={run_id}  run_status={run_status}")
print(f"Wrote 1 run row to {RUNS_TABLE} and {len(result_rows)} result rows to {RESULTS_TABLE}")

## Verify

```sql
SELECT * FROM test_automation_runs2    WHERE state_under_test = 'caseLinking_xui' ORDER BY run_start_datetime DESC;
SELECT * FROM test_automation_results2 WHERE test_from_state  = 'caseLinking_xui' ORDER BY run_id DESC;
SELECT * FROM case_link_xui_evidence   ORDER BY run_ts DESC, status;
```